# Result Availability Audit — Part 2 of 3
## Escalation events, two-clock feature matrices, and model scoring

Part 1 established that the availability interval is real: point of care results post at a median of
3 minutes, core laboratory results at 60 minutes, a twenty-fold separation within the same patients
and the same timestamp field. It also established that the record is materially incomplete when a
model actually runs, with 9.0% of drawn results still invisible at hour 6.

Part 2 turns that measurement into a scoring problem.

**What this notebook builds**

| Artefact | Content |
|---|---|
| `events_mimic.parquet`, `events_eicu.parquet` | First documented escalation of care per stay |
| `vitals_hourly_*.parquet` | Hourly vital sign aggregates, latency-insensitive |
| `grid_*.parquet` | Hourly scoring grid with labels and static covariates |
| `feat_obs_*.parquet`, `feat_avail_*.parquet` | Two-clock laboratory feature matrices |
| `comparator_*.parquet` | NEWS2 and qSOFA computed from vital signs |
| `preds_*.parquet` | Model predictions under every clock combination |
| `part2_manifest.json` | Hash-chained provenance record |

**The design**

Models are trained and scored under both clocks, giving four combinations:

|  | Scored on observation clock | Scored on availability clock |
|---|---|---|
| **Trained on observation clock** | what the literature reports | what deployment actually delivers |
| **Trained on availability clock** | not a real scenario, reported for completeness | an honestly developed model |

The off-diagonal cell in the top row is the one nobody measures. A model developed on collection
timestamps and deployed against a live record sits there, and the gap between it and the cell to its
left is the quantity this study exists to report.

Bedside scores computed from vital signs act as the comparator. Vital signs are charted at the
bedside and carry no meaningful availability delay, so a rule-based score is structurally immune to
the effect under study. If part of the apparent advantage of a learned model over a bedside score
rests on laboratory values that were not yet visible, that comparison is where it shows up.

**Runtime.** First run 25 to 60 minutes with DuckDB, considerably longer without. The dominant cost
is a single pass over `chartevents.csv`, which is larger than `labevents.csv`. Every stage caches.

**Install DuckDB before running this notebook.** Part 1 tolerated its absence. Here the ASOF joins
that build the two-clock feature matrices are roughly an order of magnitude faster with it.

---
## 1. Environment and configuration

Paths and cohort rules are inherited from Part 1 so the two notebooks cannot drift apart. Part 2 adds
only the parameters that govern scoring.

In [1]:
import sys, os, json, time, hashlib, platform, warnings, re, gc
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, asdict

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pyarrow as pa
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV
import sklearn

HAVE_DUCKDB = False
try:
    import duckdb
    HAVE_DUCKDB = True
except ImportError:
    pass

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 150)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

ENV_VERSIONS = {
    "python": platform.python_version(),
    "platform": f"{platform.system()} {platform.release()}",
    "pandas": pd.__version__, "numpy": np.__version__,
    "pyarrow": pa.__version__, "sklearn": sklearn.__version__,
    "duckdb": duckdb.__version__ if HAVE_DUCKDB else None,
}
for k, v in ENV_VERSIONS.items():
    print(f"{k:<10} {v}")
if not HAVE_DUCKDB:
    print("\n  DuckDB is absent. The ASOF joins in section 7 will fall back to pandas merge_asof,")
    print("  which works but is far slower on this volume.  pip install duckdb")

python     3.11.5
platform   Windows 10
pandas     2.3.3
numpy      1.24.4
pyarrow    24.0.0
sklearn    1.3.0
duckdb     1.5.5


In [2]:
@dataclass(frozen=True)
class Config2:
    # ---- inherited from Part 1 --------------------------------------------
    mimic_root: str = r"C:\mimic-iv-2.2"
    eicu_root:  str = r"C:\Users\kruta\Downloads\eicu-collaborative-research-database-2.0"
    out_root:   str = r"C:\Research_Paper_2\result_availability_audit"

    # ---- scoring design ----------------------------------------------------
    horizon_hours: int = 48              # scoring window after ICU admission
    prediction_horizon_hours: int = 12   # escalation within this many hours is the label
    grid_step_hours: int = 1
    min_lead_hours: float = 0.0          # blanking interval before an event; 0 = none
    min_scoring_hour: int = 6            # no scoring before a laboratory record exists
    exclude_prior_escalation: bool = True

    # ---- feature construction ---------------------------------------------
    lab_panel: tuple = (
        "creatinine", "urea_nitrogen", "sodium", "potassium", "bicarbonate",
        "chloride", "anion_gap", "glucose", "lactate", "wbc", "hemoglobin",
        "platelet", "inr", "ph", "po2", "pco2", "base_excess", "bilirubin_total",
    )
    max_staleness_hours: float = 72.0    # a value older than this is treated as absent
    vitals_window_hours: int = 6         # aggregation window for vital sign summaries

    # ---- eICU handling -----------------------------------------------------
    include_eicu: bool = True
    eicu_exclude_zero_latency_hospitals: bool = True   # from the Part 1 diagnostic
    eicu_zero_latency_median_threshold: float = 0.0

    # ---- model -------------------------------------------------------------
    train_eras: tuple = ("2008 - 2010", "2011 - 2013")
    valid_eras: tuple = ("2014 - 2016",)
    test_eras:  tuple = ("2017 - 2019",)
    hgb_max_iter: int = 2000
    hgb_learning_rate: float = 0.06
    hgb_max_leaf_nodes: int = 31
    hgb_min_samples_leaf: int = 60
    hgb_l2: float = 1.0
    early_stopping_rounds: int = 25   # retained for the manifest; superseded by the two below
    es_eval_every: int = 25           # evaluate on the stay-disjoint validation era this often
    es_patience_evals: int = 5        # stop after this many evaluations without improvement

    # ---- engine ------------------------------------------------------------
    duckdb_threads: int = 0
    duckdb_memory_limit_gb: int = 6
    pandas_chunk_rows: int = 4_000_000
    full_hash_max_gb: float = 2.0

    seed: int = 20260906
    force_rebuild: bool = False
        

CFG = Config2()

MIMIC = Path(CFG.mimic_root)
EICU  = Path(CFG.eicu_root)
OUT   = Path(CFG.out_root)

DIR = {
    "cache":  OUT / "cache",
    "tables": OUT / "tables",
    "figs":   OUT / "figures",
    "models": OUT / "models",
}
for d in DIR.values():
    d.mkdir(parents=True, exist_ok=True)

np.random.seed(CFG.seed)
CFG_JSON = json.dumps({k: (list(v) if isinstance(v, tuple) else v)
                       for k, v in asdict(CFG).items()}, sort_keys=True)
CFG_HASH = hashlib.sha256(CFG_JSON.encode()).hexdigest()

print("Output root :", OUT)
print("Config hash :", CFG_HASH[:16])
print(f"Grid: hours 1-{CFG.horizon_hours}, label = escalation within {CFG.prediction_horizon_hours} h")
print(f"Laboratory panel: {len(CFG.lab_panel)} analytes")

Output root : C:\Research_Paper_2\result_availability_audit
Config hash : e591aa162f17b4d5
Grid: hours 1-48, label = escalation within 12 h
Laboratory panel: 18 analytes


In [3]:
# ---- reproducibility scaffold, carried forward from Part 1 -----------------
class Provenance:
    def __init__(self, seed_material: str):
        self.chain = hashlib.sha256(seed_material.encode()).hexdigest()
        self.records, self.timings = [], {}

    @staticmethod
    def file_digest(path: Path, cap: int):
        size = path.stat().st_size
        h = hashlib.sha256()
        if size <= cap:
            with open(path, "rb") as f:
                for b in iter(lambda: f.read(8 << 20), b""):
                    h.update(b)
            return h.hexdigest(), "sha256_full", size
        h.update(str(size).encode())
        probe = 16 << 20
        with open(path, "rb") as f:
            for off in [0, size // 4, size // 2, (3 * size) // 4, max(0, size - probe)]:
                f.seek(off); h.update(f.read(probe))
        return h.hexdigest(), "sha256_sampled", size

    def add(self, role, name, path, extra=None):
        path = Path(path)
        if not path.exists():
            digest, method, size = "MISSING", "none", 0
        else:
            digest, method, size = self.file_digest(path, int(CFG.full_hash_max_gb * (1 << 30)))
        rec = {"role": role, "name": name, "path": str(path), "bytes": size,
               "digest": digest, "digest_method": method,
               "recorded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        if extra: rec.update(extra)
        self.chain = hashlib.sha256((self.chain + digest).encode()).hexdigest()
        rec["chain_after"] = self.chain
        self.records.append(rec)
        return rec

    def time(self, stage, seconds, detail=None):
        self.timings[stage] = {"seconds": round(seconds, 2), "detail": detail or {}}

    def manifest(self):
        return {"generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                "config": json.loads(CFG_JSON), "config_sha256": CFG_HASH,
                "environment": {**ENV_VERSIONS, "cpu_count": os.cpu_count()},
                "records": self.records, "timings": self.timings,
                "terminal_chain_sha256": self.chain}


# Chain Part 2 onto Part 1 so the two runs form one verifiable sequence.
P1_MANIFEST = OUT / "part1_manifest.json"
if P1_MANIFEST.exists():
    _p1 = json.loads(P1_MANIFEST.read_text())
    P1_CHAIN = _p1.get("terminal_chain_sha256", "")
    print(f"Part 1 terminal digest: {P1_CHAIN[:32]}...")
    print(f"Part 1 summary: {_p1.get('part1_summary', {}).get('mimic_lab_results_analytic', 'n/a'):,} "
          f"MIMIC laboratory results")
else:
    P1_CHAIN = ""
    print("WARNING: part1_manifest.json not found. Run Part 1 first; Part 2 depends on its caches.")

PROV = Provenance(CFG_HASH + P1_CHAIN)


class Stage:
    def __init__(self, label): self.label = label
    def __enter__(self):
        self.t0 = time.time(); print(f"[{self.label}] start"); return self
    def __exit__(self, *e):
        dt = time.time() - self.t0
        PROV.time(self.label, dt)
        print(f"[{self.label}] done in {dt/60:.2f} min" if dt > 90 else f"[{self.label}] done in {dt:.1f} s")
        return False


def cached(p): return Path(p).exists() and not CFG.force_rebuild

def write_table(df, name, index=False):
    p = DIR["tables"] / f"{name}.csv"; df.to_csv(p, index=index)
    PROV.add("table", name, p, {"rows": int(len(df))}); return p

def write_parquet(df, name):
    p = DIR["cache"] / f"{name}.parquet"; df.to_parquet(p, index=False, compression="zstd")
    PROV.add("cache", name, p, {"rows": int(len(df)), "cols": int(df.shape[1])}); return p

def save_fig(fig, name):
    p = DIR["figs"] / f"{name}.png"
    fig.savefig(p, dpi=200, bbox_inches="tight", facecolor="white"); plt.close(fig)
    PROV.add("figure", name, p); return p

print("Provenance chain seeded:", PROV.chain[:16])

Part 1 terminal digest: c33a334d6c4d9b9a3c109f96753b8ccf...
Part 1 summary: 6,578,347 MIMIC laboratory results
Provenance chain seeded: bb90749fc5995565


In [4]:
# ---- source resolution and engine, identical contract to Part 1 -----------
def resolve_file(root: Path, module: str, filename: str) -> Path:
    base = root / module if module else root
    stem = filename[:-4] if filename.endswith(".csv") else filename
    for c in [base / filename, base / filename / filename,
              base / f"{filename}.gz", base / filename / f"{filename}.gz",
              base / stem / filename, base / stem / f"{filename}.gz"]:
        if c.is_file():
            return c
    if base.exists():
        for pat in (f"{stem}.csv", f"{stem}.csv.gz"):
            hits = sorted(base.glob(f"**/{pat}"))
            if hits: return hits[0]
    raise FileNotFoundError(f"Could not locate '{filename}' under {base}")


NEEDED = {
    "mimic.d_items":         (MIMIC, "icu",  "d_items.csv"),
    "mimic.chartevents":     (MIMIC, "icu",  "chartevents.csv"),
    "mimic.inputevents":     (MIMIC, "icu",  "inputevents.csv"),
    "mimic.procedureevents": (MIMIC, "icu",  "procedureevents.csv"),
    "mimic.admissions":      (MIMIC, "hosp", "admissions.csv"),
}
if CFG.include_eicu:
    NEEDED.update({
        "eicu.vitalPeriodic": (EICU, "", "vitalPeriodic.csv"),
        "eicu.treatment":     (EICU, "", "treatment.csv"),
        "eicu.infusionDrug":  (EICU, "", "infusionDrug.csv"),
    })

SRC, missing = {}, []
for k, (root, mod, fn) in NEEDED.items():
    try:
        SRC[k] = resolve_file(root, mod, fn)
    except FileNotFoundError as e:
        missing.append(f"{k}: {e}")
if missing:
    print("UNRESOLVED SOURCES\n  " + "\n  ".join(missing))

inv = pd.DataFrame([{"source": k, "gb": round(p.stat().st_size / (1 << 30), 3)}
                    for k, p in SRC.items()]).sort_values("gb", ascending=False)
display(inv.reset_index(drop=True))
print(f"Total new source volume: {inv['gb'].sum():.2f} GB")


def sqlpath(p): return str(Path(p).as_posix()).replace("'", "''")

CON = None
if HAVE_DUCKDB:
    CON = duckdb.connect(":memory:")
    thr = CFG.duckdb_threads or (os.cpu_count() or 4)
    CON.execute(f"PRAGMA threads={thr}")
    CON.execute(f"PRAGMA memory_limit='{CFG.duckdb_memory_limit_gb}GB'")
    CON.execute(f"PRAGMA temp_directory='{sqlpath(DIR['cache'] / '_spill2')}'")
    print(f"\nDuckDB ready: {thr} threads, {CFG.duckdb_memory_limit_gb} GB limit")

def dsql(q): return CON.execute(q).df()

def read_csv_expr(path, **kw):
    opts = ["header=true", "sample_size=1048576", "ignore_errors=false"] + [f"{k}={v}" for k, v in kw.items()]
    return f"read_csv_auto('{sqlpath(path)}', {', '.join(opts)})"

def pandas_chunks(path, usecols=None, dtype=None, parse_dates=None):
    return pd.read_csv(path, usecols=usecols, dtype=dtype, parse_dates=parse_dates,
                       chunksize=CFG.pandas_chunk_rows, low_memory=False,
                       compression="gzip" if str(path).endswith(".gz") else "infer")

print("Engine:", "duckdb" if CON is not None else "pandas-chunked")

,source,gb
0,mimic.chartevents,28.130
1,eicu.vitalPeriodic,7.393
2,mimic.inputevents,2.185
3,eicu.treatment,0.315
4,eicu.infusionDrug,0.260
5,mimic.procedureevents,0.121
6,mimic.admissions,0.068
7,mimic.d_items,0.000


Total new source volume: 38.47 GB

DuckDB ready: 14 threads, 6 GB limit
Engine: duckdb


---
## 2. Part 1 artefacts

The cohorts and the laboratory extraction are loaded rather than rebuilt. The eICU zero-latency
diagnostic from Part 1 is applied here as a cohort restriction: hospitals whose revised-result offset
is never populated cannot contribute a meaningful availability clock, and including them would let a
data contribution artefact masquerade as instant result release.

In [5]:
with Stage("load-part1"):
    cohort_m = pd.read_parquet(DIR["cache"] / "cohort_mimic.parquet")
    labs_m   = pd.read_parquet(DIR["cache"] / "labs_mimic.parquet")
    if CFG.include_eicu:
        cohort_e = pd.read_parquet(DIR["cache"] / "cohort_eicu.parquet")
        labs_e   = pd.read_parquet(DIR["cache"] / "labs_eicu.parquet")

# Part 1 wrote the original turnaround class; re-derive the blood gas correction so the two
# notebooks agree without depending on the order cells were run in.
dlab = pd.read_csv(resolve_file(MIMIC, "hosp", "d_labitems.csv"))
dlab.columns = [c.lower() for c in dlab.columns]
cat_map = dlab.assign(itemid=lambda d: d["itemid"].astype(int)).set_index("itemid")["category"]
labs_m["itemid"] = labs_m["itemid"].astype(int)
labs_m["category"] = labs_m["itemid"].map(cat_map)
labs_m.loc[labs_m["category"].eq("Blood Gas"), "turnaround_class"] = "poc"

# Analytic latency window, matching Part 1.
labs_m = labs_m[labs_m["latency_min"].notna() &
                (labs_m["latency_min"] >= 0) & (labs_m["latency_min"] <= 1440)].copy()
labs_m["latency_h"] = labs_m["latency_min"] / 60.0

print(f"MIMIC-IV cohort  : {len(cohort_m):,} stays")
print(f"MIMIC-IV labs    : {len(labs_m):,} analytic results")

if CFG.include_eicu:
    labs_e = labs_e[labs_e["latency_min"].notna() &
                    (labs_e["latency_min"] >= 0) & (labs_e["latency_min"] <= 1440)].copy()
    labs_e["latency_h"] = labs_e["latency_min"] / 60.0

    if CFG.eicu_exclude_zero_latency_hospitals:
        hz = labs_e.groupby("hospitalid")["latency_min"].median()
        drop_h = set(hz.index[hz <= CFG.eicu_zero_latency_median_threshold])
        keep_frac = 1 - len(drop_h) / max(len(hz), 1)
        print(f"\neICU restriction: {len(drop_h)} of {len(hz)} hospitals never populate the")
        print("  revised-result offset (median latency of zero) and cannot support an")
        print("  availability clock.")
        if keep_frac < 0.10:
            print("  WARNING: this would remove more than 90% of hospitals. The exclusion is being")
            print("  skipped and eICU is retained without an availability clock. Do not report an")
            print("  eICU latency estimate under this condition.")
        else:
            n_before = len(cohort_e)
            cohort_e = cohort_e[~cohort_e["hospitalid"].isin(drop_h)].copy()
            labs_e   = labs_e[~labs_e["hospitalid"].isin(drop_h)].copy()
            print(f"  Stays {n_before:,} -> {len(cohort_e):,}; hospitals remaining "
                  f"{cohort_e['hospitalid'].nunique()}")
            write_table(pd.DataFrame({"hospitalid": sorted(drop_h)}), "t20_eicu_excluded_hospitals")

    print(f"eICU cohort      : {len(cohort_e):,} stays across {cohort_e['hospitalid'].nunique()} hospitals")
    print(f"eICU labs        : {len(labs_e):,} analytic results")

[load-part1] start
[load-part1] done in 6.2 s
MIMIC-IV cohort  : 48,736 stays
MIMIC-IV labs    : 6,578,347 analytic results

eICU restriction: 43 of 206 hospitals never populate the
  revised-result offset (median latency of zero) and cannot support an
  availability clock.
  Stays 120,354 -> 92,342; hospitals remaining 165
eICU cohort      : 92,342 stays across 165 hospitals
eICU labs        : 7,124,093 analytic results


---
## 3. Escalation of care

The clinical endpoint is the first documented escalation of care during the ICU stay, defined as the
earliest of vasopressor initiation, initiation of invasive ventilation, or initiation of renal
replacement therapy. These are acts a clinician performed and documented, not derived risk states, and
they are the decisions an early warning system exists to anticipate.

Item identifiers are resolved by matching each database's own dictionary, and the resolved mapping is
printed. Hard-coded identifier lists are not trusted.

In [6]:
# ---- resolve MIMIC-IV item identifiers from d_items ------------------------
ditems = pd.read_csv(SRC["mimic.d_items"])
ditems.columns = [c.lower() for c in ditems.columns]
ditems["label_l"] = ditems["label"].astype(str).str.lower().str.strip()

EVENT_PATTERNS = {
    "vasopressor": ("inputevents",
        r"^(norepinephrine|epinephrine|phenylephrine|vasopressin|dopamine)$"),
    "ventilation": ("procedureevents", r"^invasive ventilation$"),
    "rrt":         ("procedureevents",
        r"(crrt|cvvh|hemodialysis|dialysis - c|peritoneal dialysis|dialysis catheter)"),
}

ev_map = []
for ev, (link, rx) in EVENT_PATTERNS.items():
    sel = ditems["label_l"].str.contains(rx, regex=True, na=False)
    if "linksto" in ditems.columns:
        sel &= ditems["linksto"].astype(str).str.lower().eq(link)
    for _, r in ditems[sel].iterrows():
        ev_map.append({"event": ev, "linksto": link, "itemid": int(r["itemid"]),
                       "label": r["label"], "category": r.get("category", "")})

EV_MAP = pd.DataFrame(ev_map).drop_duplicates("itemid").reset_index(drop=True)
display(EV_MAP.sort_values(["event", "label"]).reset_index(drop=True))
write_table(EV_MAP, "t21_escalation_item_map_mimic")

for ev in EVENT_PATTERNS:
    n = (EV_MAP["event"] == ev).sum()
    print(f"{ev:<14} {n:>3} itemids" + ("   <-- NONE RESOLVED, CHECK PATTERN" if n == 0 else ""))

,event,linksto,itemid,label,category
0,rrt,procedureevents,225436,CRRT Filter Change,Dialysis
1,rrt,procedureevents,225802,Dialysis - CRRT,Dialysis
2,rrt,procedureevents,225803,Dialysis - CVVHD,Dialysis
3,rrt,procedureevents,225809,Dialysis - CVVHDF,Dialysis
4,rrt,procedureevents,224270,Dialysis Catheter,Access Lines - Invasive
5,rrt,procedureevents,225441,Hemodialysis,4-Procedures
6,rrt,procedureevents,225805,Peritoneal Dialysis,Dialysis
7,vasopressor,inputevents,221662,Dopamine,Medications
8,vasopressor,inputevents,221289,Epinephrine,Medications
9,vasopressor,inputevents,221906,Norepinephrine,Medications


vasopressor      5 itemids
ventilation      1 itemids
rrt              7 itemids


In [7]:
EVENTS_M_P = DIR["cache"] / "events_mimic.parquet"

with Stage("events-mimic"):
    if cached(EVENTS_M_P):
        events_m = pd.read_parquet(EVENTS_M_P)
        print("loaded from cache")
    else:
        inp_ids = EV_MAP.loc[EV_MAP["linksto"] == "inputevents", "itemid"].astype(int).tolist()
        prc_ids = EV_MAP.loc[EV_MAP["linksto"] == "procedureevents", "itemid"].astype(int).tolist()
        keys = cohort_m[["stay_id", "subject_id", "intime", "outtime"]]

        def _window(df, tcol):
            j = df.merge(keys, on="stay_id", how="inner")
            j = j[(j[tcol] >= j["intime"]) & (j[tcol] <= j["outtime"])]
            j["event_h"] = (j[tcol] - j["intime"]).dt.total_seconds() / 3600.0
            return j

        if CON is not None:
            CON.register("coh", keys)
            CON.register("ev_map_tbl", EV_MAP[["itemid", "event"]])
            parts = []
            if inp_ids:
                parts.append(dsql(f"""
                    SELECT c.stay_id, 'vasopressor' AS event,
                           DATE_DIFF('minute', c.intime, TRY_CAST(i.starttime AS TIMESTAMP))/60.0 AS event_h
                    FROM {read_csv_expr(SRC['mimic.inputevents'])} i
                    JOIN coh c ON TRY_CAST(i.stay_id AS BIGINT) = c.stay_id
                    WHERE TRY_CAST(i.itemid AS INTEGER) IN ({','.join(map(str, inp_ids))})
                      AND TRY_CAST(i.starttime AS TIMESTAMP) BETWEEN c.intime AND c.outtime
                      AND TRY_CAST(i.rate AS DOUBLE) > 0
                """))
            if prc_ids:
                parts.append(dsql(f"""
                    SELECT c.stay_id, m.event AS event,
                           DATE_DIFF('minute', c.intime, TRY_CAST(p.starttime AS TIMESTAMP))/60.0 AS event_h
                    FROM {read_csv_expr(SRC['mimic.procedureevents'])} p
                    JOIN coh c ON TRY_CAST(p.stay_id AS BIGINT) = c.stay_id
                    JOIN ev_map_tbl m ON TRY_CAST(p.itemid AS INTEGER) = m.itemid
                    WHERE TRY_CAST(p.starttime AS TIMESTAMP) BETWEEN c.intime AND c.outtime
                """))
            raw = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
                columns=["stay_id", "event", "event_h"])
        else:
            parts = []
            if inp_ids:
                for ch in pandas_chunks(SRC["mimic.inputevents"],
                                        usecols=["stay_id", "itemid", "starttime", "rate"],
                                        parse_dates=["starttime"]):
                    ch = ch[ch["itemid"].isin(inp_ids) & (ch["rate"].fillna(0) > 0)]
                    if len(ch):
                        w = _window(ch, "starttime")
                        parts.append(w.assign(event="vasopressor")[["stay_id", "event", "event_h"]])
            if prc_ids:
                pe = pd.read_csv(SRC["mimic.procedureevents"],
                                 usecols=["stay_id", "itemid", "starttime"], parse_dates=["starttime"])
                pe = pe[pe["itemid"].isin(prc_ids)]
                w = _window(pe, "starttime").merge(EV_MAP[["itemid", "event"]], on="itemid", how="left")
                parts.append(w[["stay_id", "event", "event_h"]])
            raw = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
                columns=["stay_id", "event", "event_h"])

        raw = raw[raw["event_h"].notna() & (raw["event_h"] >= 0)]
        first_any = raw.groupby("stay_id")["event_h"].min().rename("first_event_h")
        first_typ = (raw.sort_values("event_h").groupby("stay_id")["event"].first()
                     .rename("first_event_type"))
        by_type = raw.groupby(["stay_id", "event"])["event_h"].min().unstack()
        by_type.columns = [f"first_{c}_h" for c in by_type.columns]

        events_m = (cohort_m[["stay_id"]]
                    .merge(first_any, on="stay_id", how="left")
                    .merge(first_typ, on="stay_id", how="left")
                    .merge(by_type.reset_index(), on="stay_id", how="left"))
        write_parquet(events_m, "events_mimic")

n_ev = events_m["first_event_h"].notna().sum()
print(f"\nStays with a documented escalation: {n_ev:,} / {len(events_m):,} ({100*n_ev/len(events_m):.1f}%)")
print(f"Within the first {CFG.horizon_hours} h: "
      f"{(events_m['first_event_h'] <= CFG.horizon_hours).sum():,}")
display(events_m["first_event_type"].value_counts(dropna=False).rename("stays"))
print("\nTime to first escalation (hours):")
display(events_m["first_event_h"].describe(percentiles=[.1, .25, .5, .75, .9]))

[events-mimic] start
loaded from cache
[events-mimic] done in 0.3 s

Stays with a documented escalation: 23,161 / 48,736 (47.5%)
Within the first 48 h: 22,441


first_event_type
None           25575
ventilation    14226
vasopressor     8105
rrt              830
Name: stays, dtype: int64


Time to first escalation (hours):


count   23,161.000
mean         6.900
std         20.707
min          0.000
10%          0.133
25%          0.400
50%          1.883
75%          4.367
90%         13.450
max        507.467
Name: first_event_h, dtype: float64

In [8]:
EVENTS_E_P = DIR["cache"] / "events_eicu.parquet"

if CFG.include_eicu:
    with Stage("events-eicu"):
        if cached(EVENTS_E_P):
            events_e = pd.read_parquet(EVENTS_E_P)
            print("loaded from cache")
        else:
            # eICU records interventions as hierarchical treatment strings and infusion drug names.
            TREAT_RX = {
                "vasopressor": r"vasopressor|norepinephrine|epinephrine|phenylephrine|vasopressin|dopamine",
                "ventilation": r"mechanical ventilation|intubation|ventilator",
                "rrt":         r"dialysis|cvvh|crrt|ultrafiltration",
            }
            INFUSE_RX = r"(norepinephrine|levophed|epinephrine|phenylephrine|neo-synephrine|vasopressin|dopamine)"

            keys_e = cohort_e[["patientunitstayid", "unitdischargeoffset"]]
            parts = []

            if CON is not None:
                CON.register("coh_e", keys_e)
                tr = dsql(f"""
                    SELECT TRY_CAST(t.patientunitstayid AS BIGINT) AS patientunitstayid,
                           LOWER(CAST(t.treatmentstring AS VARCHAR)) AS s,
                           TRY_CAST(t.treatmentoffset AS DOUBLE)/60.0 AS event_h
                    FROM {read_csv_expr(SRC['eicu.treatment'])} t
                    JOIN coh_e c ON TRY_CAST(t.patientunitstayid AS BIGINT) = c.patientunitstayid
                    WHERE TRY_CAST(t.treatmentoffset AS DOUBLE) BETWEEN 0 AND c.unitdischargeoffset
                """)
                inf = dsql(f"""
                    SELECT TRY_CAST(d.patientunitstayid AS BIGINT) AS patientunitstayid,
                           LOWER(CAST(d.drugname AS VARCHAR)) AS s,
                           TRY_CAST(d.infusionoffset AS DOUBLE)/60.0 AS event_h
                    FROM {read_csv_expr(SRC['eicu.infusionDrug'])} d
                    JOIN coh_e c ON TRY_CAST(d.patientunitstayid AS BIGINT) = c.patientunitstayid
                    WHERE TRY_CAST(d.infusionoffset AS DOUBLE) BETWEEN 0 AND c.unitdischargeoffset
                      AND TRY_CAST(d.drugrate AS DOUBLE) > 0
                """)
            else:
                acc_t, acc_i = [], []
                idx = keys_e.set_index("patientunitstayid")
                for ch in pandas_chunks(SRC["eicu.treatment"],
                                        usecols=["patientunitstayid", "treatmentoffset", "treatmentstring"]):
                    ch = ch.join(idx, on="patientunitstayid", how="inner")
                    ch = ch[(ch["treatmentoffset"] >= 0) &
                            (ch["treatmentoffset"] <= ch["unitdischargeoffset"])]
                    if len(ch):
                        acc_t.append(pd.DataFrame({
                            "patientunitstayid": ch["patientunitstayid"].values,
                            "s": ch["treatmentstring"].astype(str).str.lower().values,
                            "event_h": ch["treatmentoffset"].values / 60.0}))
                for ch in pandas_chunks(SRC["eicu.infusionDrug"],
                                        usecols=["patientunitstayid", "infusionoffset", "drugname", "drugrate"]):
                    ch["drugrate"] = pd.to_numeric(ch["drugrate"], errors="coerce")
                    ch = ch.join(idx, on="patientunitstayid", how="inner")
                    ch = ch[(ch["infusionoffset"] >= 0) &
                            (ch["infusionoffset"] <= ch["unitdischargeoffset"]) &
                            (ch["drugrate"].fillna(0) > 0)]
                    if len(ch):
                        acc_i.append(pd.DataFrame({
                            "patientunitstayid": ch["patientunitstayid"].values,
                            "s": ch["drugname"].astype(str).str.lower().values,
                            "event_h": ch["infusionoffset"].values / 60.0}))
                tr = pd.concat(acc_t, ignore_index=True) if acc_t else pd.DataFrame(columns=["patientunitstayid","s","event_h"])
                inf = pd.concat(acc_i, ignore_index=True) if acc_i else pd.DataFrame(columns=["patientunitstayid","s","event_h"])

            for ev, rx in TREAT_RX.items():
                hit = tr[tr["s"].str.contains(rx, regex=True, na=False)]
                if len(hit):
                    parts.append(hit.assign(event=ev)[["patientunitstayid", "event", "event_h"]])
            hit = inf[inf["s"].str.contains(INFUSE_RX, regex=True, na=False)]
            if len(hit):
                parts.append(hit.assign(event="vasopressor")[["patientunitstayid", "event", "event_h"]])

            raw_e = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
                columns=["patientunitstayid", "event", "event_h"])
            raw_e = raw_e[raw_e["event_h"].notna() & (raw_e["event_h"] >= 0)]

            fa = raw_e.groupby("patientunitstayid")["event_h"].min().rename("first_event_h")
            ft = (raw_e.sort_values("event_h").groupby("patientunitstayid")["event"].first()
                  .rename("first_event_type"))
            bt = raw_e.groupby(["patientunitstayid", "event"])["event_h"].min().unstack()
            bt.columns = [f"first_{c}_h" for c in bt.columns]

            events_e = (cohort_e[["patientunitstayid"]]
                        .merge(fa, on="patientunitstayid", how="left")
                        .merge(ft, on="patientunitstayid", how="left")
                        .merge(bt.reset_index(), on="patientunitstayid", how="left"))
            write_parquet(events_e, "events_eicu")
            del tr, inf, raw_e; gc.collect()

    n = events_e["first_event_h"].notna().sum()
    print(f"\neICU stays with a documented escalation: {n:,} / {len(events_e):,} ({100*n/len(events_e):.1f}%)")
    display(events_e["first_event_type"].value_counts(dropna=False).rename("stays"))
else:
    events_e = pd.DataFrame()
    print("eICU disabled in configuration.")

[events-eicu] start
loaded from cache
[events-eicu] done in 0.0 s

eICU stays with a documented escalation: 31,021 / 92,342 (33.6%)


first_event_type
None           61321
ventilation    17082
vasopressor    11738
rrt             2201
Name: stays, dtype: int64

In [9]:
# ---- side-by-side endpoint description ------------------------------------
def endpoint_row(ev, coh, db, idcol):
    n = len(coh)
    has = ev["first_event_h"].notna()
    early = ev["first_event_h"] <= CFG.horizon_hours
    d = {"database": db, "stays": f"{n:,}",
         "any escalation, n (%)": f"{has.sum():,} ({100*has.mean():.1f}%)",
         f"escalation within {CFG.horizon_hours} h, n (%)": f"{early.sum():,} ({100*early.mean():.1f}%)",
         "time to escalation h, median (IQR)":
             f"{ev['first_event_h'].median():.1f} "
             f"({ev['first_event_h'].quantile(.25):.1f}-{ev['first_event_h'].quantile(.75):.1f})"}
    vc = ev["first_event_type"].value_counts()
    for t in ("vasopressor", "ventilation", "rrt"):
        d[f"first event = {t}, n"] = f"{int(vc.get(t, 0)):,}"
    return d

rows = [endpoint_row(events_m, cohort_m, "MIMIC-IV", "stay_id")]
if CFG.include_eicu:
    rows.append(endpoint_row(events_e, cohort_e, "eICU-CRD", "patientunitstayid"))
tbl_ep = pd.DataFrame(rows).set_index("database").T
display(tbl_ep)
write_table(tbl_ep, "t22_escalation_endpoint", index=True)

database,MIMIC-IV,eICU-CRD
stays,"48,736","92,342"
"any escalation, n (%)","23,161 (47.5%)","31,021 (33.6%)"
"escalation within 48 h, n (%)","22,441 (46.0%)","29,963 (32.4%)"
"time to escalation h, median (IQR)",1.9 (0.4-4.4),1.2 (0.5-3.8)
"first event = vasopressor, n","8,105","11,738"
"first event = ventilation, n","14,226","17,082"
"first event = rrt, n",830,"2,201"


WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t22_escalation_endpoint.csv')

---
## 4. Vital signs

Vital signs are charted at the bedside and reach the record essentially at the moment of measurement,
so they are treated as latency-insensitive throughout. That property is what makes them a valid
comparator: a bedside score is structurally immune to the effect under study, and any part of a
learned model's advantage that rests on delayed laboratory values will show up as a narrowing of the
gap between the two under the availability clock.

`chartevents.csv` is the largest file in either database. It is aggregated to hourly summaries during
the scan rather than materialised in full, which keeps memory bounded regardless of engine.

In [10]:
# ---- resolve vital sign itemids -------------------------------------------
VITAL_PATTERNS = {
    "hr":     r"^heart rate$",
    "sbp":    r"^(arterial blood pressure systolic|non invasive blood pressure systolic)$",
    "dbp":    r"^(arterial blood pressure diastolic|non invasive blood pressure diastolic)$",
    "map":    r"^(arterial blood pressure mean|non invasive blood pressure mean)$",
    "rr":     r"^respiratory rate$",
    "spo2":   r"^o2 saturation pulseoxymetry$",
    "temp_c": r"^temperature celsius$",
    "temp_f": r"^temperature fahrenheit$",
    "gcs_eye": r"^gcs - eye opening$",
    "gcs_verbal": r"^gcs - verbal response$",
    "gcs_motor": r"^gcs - motor response$",
    "fio2":   r"^inspired o2 fraction$",
}

rows = []
for vit, rx in VITAL_PATTERNS.items():
    sel = ditems["label_l"].str.match(rx, na=False)
    if "linksto" in ditems.columns:
        sel &= ditems["linksto"].astype(str).str.lower().eq("chartevents")
    for _, r in ditems[sel].iterrows():
        rows.append({"vital": vit, "itemid": int(r["itemid"]), "label": r["label"]})

VIT_MAP = pd.DataFrame(rows).drop_duplicates("itemid").reset_index(drop=True)
VIT_IDS = VIT_MAP["itemid"].astype(int).tolist()
display(VIT_MAP.sort_values(["vital", "itemid"]).reset_index(drop=True))
write_table(VIT_MAP, "t23_vital_item_map_mimic")

missing_v = sorted(set(VITAL_PATTERNS) - set(VIT_MAP["vital"]))
print(f"{len(VIT_IDS)} itemids for {VIT_MAP['vital'].nunique()} of {len(VITAL_PATTERNS)} vitals")
if missing_v:
    print("  unresolved:", ", ".join(missing_v))

,vital,itemid,label
0,dbp,220051,Arterial Blood Pressure diastolic
1,dbp,220180,Non Invasive Blood Pressure diastolic
2,fio2,223835,Inspired O2 Fraction
3,gcs_eye,220739,GCS - Eye Opening
4,gcs_motor,223901,GCS - Motor Response
5,gcs_verbal,223900,GCS - Verbal Response
6,hr,220045,Heart Rate
7,map,220052,Arterial Blood Pressure mean
8,map,220181,Non Invasive Blood Pressure mean
9,rr,220210,Respiratory Rate


15 itemids for 12 of 12 vitals


In [11]:
VITALS_M_P = DIR["cache"] / "vitals_hourly_mimic.parquet"

with Stage("vitals-mimic"):
    if cached(VITALS_M_P):
        vit_m = pd.read_parquet(VITALS_M_P)
        print("loaded from cache")
    else:
        keys = cohort_m[["stay_id", "intime", "outtime"]]
        H = CFG.horizon_hours
        if CON is not None:
            CON.register("coh_v", keys)
            CON.register("vit_map", VIT_MAP[["itemid", "vital"]])
            vit_m = dsql(f"""
                SELECT c.stay_id, v.vital,
                       CAST(FLOOR(DATE_DIFF('minute', c.intime,
                            TRY_CAST(ce.charttime AS TIMESTAMP))/60.0) AS INTEGER) AS hour,
                       AVG(TRY_CAST(ce.valuenum AS DOUBLE)) AS mean_v,
                       MIN(TRY_CAST(ce.valuenum AS DOUBLE)) AS min_v,
                       MAX(TRY_CAST(ce.valuenum AS DOUBLE)) AS max_v,
                       COUNT(*) AS n_v
                FROM {read_csv_expr(SRC['mimic.chartevents'])} ce
                JOIN coh_v c ON TRY_CAST(ce.stay_id AS BIGINT) = c.stay_id
                JOIN vit_map v ON TRY_CAST(ce.itemid AS INTEGER) = v.itemid
                WHERE TRY_CAST(ce.valuenum AS DOUBLE) IS NOT NULL
                  AND TRY_CAST(ce.charttime AS TIMESTAMP) >= c.intime
                  AND DATE_DIFF('minute', c.intime, TRY_CAST(ce.charttime AS TIMESTAMP)) <= {H * 60}
                GROUP BY 1, 2, 3
            """)
        else:
            keep = set(VIT_IDS)
            vmap = VIT_MAP.set_index("itemid")["vital"]
            idx = keys.set_index("stay_id")
            acc = []
            for i, ch in enumerate(pandas_chunks(
                    SRC["mimic.chartevents"],
                    usecols=["stay_id", "itemid", "charttime", "valuenum"],
                    parse_dates=["charttime"])):
                ch = ch[ch["itemid"].isin(keep) & ch["valuenum"].notna()]
                if ch.empty:
                    continue
                ch = ch.join(idx, on="stay_id", how="inner")
                ch["hour"] = np.floor((ch["charttime"] - ch["intime"]).dt.total_seconds() / 3600.0)
                ch = ch[(ch["hour"] >= 0) & (ch["hour"] <= H)]
                if ch.empty:
                    continue
                ch["vital"] = ch["itemid"].map(vmap)
                g = (ch.groupby(["stay_id", "vital", "hour"])["valuenum"]
                     .agg(mean_v="mean", min_v="min", max_v="max", n_v="size").reset_index())
                acc.append(g)
                if i % 10 == 0:
                    print(f"  chunk {i}, hourly rows so far {sum(len(a) for a in acc):,}")
            part = pd.concat(acc, ignore_index=True) if acc else pd.DataFrame()
            # chunk boundaries can split an hour, so recombine
            vit_m = (part.assign(_s=part["mean_v"] * part["n_v"])
                     .groupby(["stay_id", "vital", "hour"])
                     .agg(_s=("_s", "sum"), n_v=("n_v", "sum"),
                          min_v=("min_v", "min"), max_v=("max_v", "max")).reset_index())
            vit_m["mean_v"] = vit_m["_s"] / vit_m["n_v"]
            vit_m = vit_m.drop(columns=["_s"])

        vit_m["hour"] = vit_m["hour"].astype("int16")
        for c in ("mean_v", "min_v", "max_v"):
            vit_m[c] = vit_m[c].astype("float32")
        vit_m["n_v"] = vit_m["n_v"].astype("int32")
        write_parquet(vit_m, "vitals_hourly_mimic")

print(f"MIMIC-IV hourly vital rows: {len(vit_m):,} across {vit_m['stay_id'].nunique():,} stays")
display(vit_m.groupby("vital")["n_v"].agg(hourly_cells="size", observations="sum"))

[vitals-mimic] start
loaded from cache
[vitals-mimic] done in 1.5 s
MIMIC-IV hourly vital rows: 12,478,989 across 48,690 stays


,hourly_cells,observations
vital,,
dbp,1643425,2042101
fio2,198642,208943
gcs_eye,552465,561918
gcs_motor,550457,559692
gcs_verbal,551762,561077
hr,1738536,2056474
map,1644874,2043656
rr,1720959,2033468
sbp,1643637,2042494


In [12]:
VITALS_E_P = DIR["cache"] / "vitals_hourly_eicu.parquet"

if CFG.include_eicu:
    with Stage("vitals-eicu"):
        if cached(VITALS_E_P):
            vit_e = pd.read_parquet(VITALS_E_P)
            print("loaded from cache")
        else:
            # vitalPeriodic is wide: one row per observation with a column per signal.
            E_COLS = {"heartrate": "hr", "systemicsystolic": "sbp", "systemicdiastolic": "dbp",
                      "systemicmean": "map", "respiration": "rr", "sao2": "spo2",
                      "temperature": "temp_c"}
            H = CFG.horizon_hours
            keys_e = cohort_e[["patientunitstayid"]]
            if CON is not None:
                CON.register("coh_ve", keys_e)
                sel = ", ".join(f"TRY_CAST(v.{c} AS DOUBLE) AS {n}" for c, n in E_COLS.items())
                wide = dsql(f"""
                    SELECT c.patientunitstayid,
                           CAST(FLOOR(TRY_CAST(v.observationoffset AS DOUBLE)/60.0) AS INTEGER) AS hour,
                           {sel}
                    FROM {read_csv_expr(SRC['eicu.vitalPeriodic'])} v
                    JOIN coh_ve c ON TRY_CAST(v.patientunitstayid AS BIGINT) = c.patientunitstayid
                    WHERE TRY_CAST(v.observationoffset AS DOUBLE) BETWEEN 0 AND {H * 60}
                """)
            else:
                idx = keys_e.assign(_k=1).set_index("patientunitstayid")
                acc = []
                for i, ch in enumerate(pandas_chunks(
                        SRC["eicu.vitalPeriodic"],
                        usecols=["patientunitstayid", "observationoffset"] + list(E_COLS))):
                    ch = ch.join(idx, on="patientunitstayid", how="inner")
                    ch = ch[(ch["observationoffset"] >= 0) & (ch["observationoffset"] <= H * 60)]
                    if ch.empty:
                        continue
                    ch["hour"] = np.floor(ch["observationoffset"] / 60.0)
                    acc.append(ch.rename(columns=E_COLS)[
                        ["patientunitstayid", "hour"] + list(E_COLS.values())])
                    if i % 5 == 0:
                        print(f"  chunk {i}, rows so far {sum(len(a) for a in acc):,}")
                wide = pd.concat(acc, ignore_index=True) if acc else pd.DataFrame()

            long = wide.melt(id_vars=["patientunitstayid", "hour"],
                             var_name="vital", value_name="v").dropna(subset=["v"])
            vit_e = (long.groupby(["patientunitstayid", "vital", "hour"])["v"]
                     .agg(mean_v="mean", min_v="min", max_v="max", n_v="size").reset_index())
            vit_e["hour"] = vit_e["hour"].astype("int16")
            for c in ("mean_v", "min_v", "max_v"):
                vit_e[c] = vit_e[c].astype("float32")
            vit_e["n_v"] = vit_e["n_v"].astype("int32")
            write_parquet(vit_e, "vitals_hourly_eicu")
            del wide, long; gc.collect()

    print(f"eICU hourly vital rows: {len(vit_e):,} across {vit_e['patientunitstayid'].nunique():,} stays")
else:
    vit_e = pd.DataFrame()

[vitals-eicu] start
loaded from cache
[vitals-eicu] done in 1.2 s
eICU hourly vital rows: 11,675,383 across 91,535 stays


---
## 5. Scoring grid

One row per stay per hour, which is how a deployed early warning system actually runs.

A row is retained while the patient is still in the ICU and has not yet escalated. Hours after the
first escalation are dropped: once the intervention has happened the prediction question no longer
exists. The label is whether the first escalation falls in the next `prediction_horizon_hours`.

In [13]:
def build_grid(cohort, events, idcol, los_col, horizon, pred_h,
               min_hour=1, exclude_prior=True):
    base = cohort[[idcol, los_col]].copy()
    base["n_hours"] = np.minimum(np.floor(base[los_col]).astype(int), horizon)
    base = base[base["n_hours"] >= min_hour]

    # A patient already receiving the intervention is not at risk of starting it.
    # Without this, stays that arrive intubated are scored as deterioration events.
    if exclude_prior:
        early = set(events.loc[events["first_event_h"].notna() &
                               (events["first_event_h"] < min_hour), idcol])
        n0 = len(base)
        base = base[~base[idcol].isin(early)]
        print(f"  excluded {n0 - len(base):,} stays already escalated before hour {min_hour}")

    if len(base) == 0:
        return pd.DataFrame(columns=[idcol, "hour", "first_event_h", "first_event_type",
                                     "y", "hours_to_event"])

    nh = base["n_hours"].to_numpy()
    grid = pd.DataFrame({
        idcol: np.repeat(base[idcol].to_numpy(), nh - min_hour + 1),
        "hour": np.concatenate([np.arange(min_hour, n + 1) for n in nh]),
    })
    grid = grid.merge(events[[idcol, "first_event_h", "first_event_type"]], on=idcol, how="left")

    # drop hours at or after the escalation itself
    ev0 = grid["first_event_h"]
    grid = grid[ev0.isna() | (grid["hour"] < ev0)].copy().reset_index(drop=True)

    # label: escalation occurs within the prediction window
    ev = grid["first_event_h"]
    grid["y"] = ((ev.notna()) & (ev > grid["hour"]) &
                 (ev <= grid["hour"] + pred_h)).astype("int8")
    grid["hours_to_event"] = np.where(ev.notna(), ev - grid["hour"], np.nan)
    grid["hour"] = grid["hour"].astype("int16")
    return grid.reset_index(drop=True)


with Stage("grid-build"):
    cohort_m["_los_h"] = cohort_m["icu_hours"]
    grid_m = build_grid(cohort_m, events_m, "stay_id", "_los_h",
                        CFG.horizon_hours, CFG.prediction_horizon_hours,
                        CFG.min_scoring_hour, CFG.exclude_prior_escalation)
    if CFG.include_eicu:
        cohort_e["_los_h"] = cohort_e["icu_hours"]
        grid_e = build_grid(cohort_e, events_e, "patientunitstayid", "_los_h",
                            CFG.horizon_hours, CFG.prediction_horizon_hours,
                            CFG.min_scoring_hour, CFG.exclude_prior_escalation)

def grid_row(g, idcol, db):
    return {"database": db, "scoring hours": f"{len(g):,}",
            "stays": f"{g[idcol].nunique():,}",
            "positive hours, n (%)": f"{int(g['y'].sum()):,} ({100*g['y'].mean():.2f}%)",
            "stays with >=1 positive hour": f"{g.loc[g['y'] == 1, idcol].nunique():,}",
            "hours per stay, median": f"{g.groupby(idcol).size().median():.0f}"}

rows = [grid_row(grid_m, "stay_id", "MIMIC-IV")]
if CFG.include_eicu:
    rows.append(grid_row(grid_e, "patientunitstayid", "eICU-CRD"))
tbl_g = pd.DataFrame(rows).set_index("database").T
display(tbl_g)
write_table(tbl_g, "t24_scoring_grid", index=True)

[grid-build] start
  excluded 19,088 stays already escalated before hour 6
  excluded 25,166 stays already escalated before hour 6
[grid-build] done in 1.3 s


database,MIMIC-IV,eICU-CRD
scoring hours,"823,501","1,892,310"
stays,"29,635","67,167"
"positive hours, n (%)","24,718 (3.00%)","39,134 (2.07%)"
stays with >=1 positive hour,"3,497","5,041"
"hours per stay, median",28,29


WindowsPath('C:/Research_Paper_2/result_availability_audit/tables/t24_scoring_grid.csv')

---
## 6. Two-clock laboratory features

This is the heart of the study.

For every scoring hour the same feature is constructed twice from the same underlying results. The
observation clock asks what had been *drawn* by that hour. The availability clock asks what had been
*released* by that hour. The difference between the two matrices is the entire deployment gap.

Three features per analyte per clock:

* **value** — the most recent result available under that clock
* **staleness** — hours since that result. Under the availability clock this is measured from the
  release time, which is what a clinician looking at the chart would see
* **delta** — change from the preceding result, which is what carries trajectory

Staleness is worth attention in its own right. It is not merely a nuisance covariate: under the
availability clock a model's most recent creatinine is systematically older than the literature
assumes, and that shift is invisible unless both clocks are constructed side by side.

In [14]:
def asof_features(grid, labs, idcol, select_col, panel, tag, max_stale, age_col="obs_h"):
    """
    Last-value-carried-forward under one clock, with staleness and delta.
    `select_col` decides which results are eligible at a scoring hour: the draw time under
    the observation clock, the release time under the availability clock. `age_col` is always
    the draw time, so staleness is the age of the specimen under both clocks. That is the
    clinically meaningful quantity: a creatinine drawn six hours ago is six hours old whether
    it posted an hour ago or immediately, and the availability clock forces the model back
    onto an older specimen rather than onto a fresher one.

    An explicit row identifier is carried through the join so alignment never depends on row
    ordering. DuckDB ASOF JOIN when available, pandas merge_asof otherwise.
    """
    out = grid[[idcol, "hour"]].copy().reset_index(drop=True)
    out["_rid"] = np.arange(len(out), dtype="int64")
    out["_t"] = out["hour"].astype("float64")
    left = out[["_rid", idcol, "_t"]]

    for a in panel:
        cols = [idcol, select_col, "valuenum"] + ([age_col] if age_col != select_col else [])
        sub = labs.loc[labs["analyte"] == a, cols].dropna()
        if sub.empty:
            for suf in ("val", "stale", "delta"):
                out[f"{a}_{suf}_{tag}"] = np.nan
            continue
        sub = sub.rename(columns={select_col: "_e"})
        sub["_age"] = sub[age_col] if age_col != select_col else sub["_e"]
        sub = sub.sort_values([idcol, "_e"])
        sub["_prev"] = sub.groupby(idcol)["valuenum"].shift(1)
        sub = sub[[idcol, "_e", "_age", "valuenum", "_prev"]]

        if CON is not None:
            CON.register("g_", left)
            CON.register("s_", sub)
            m = CON.execute(f"""
                SELECT g._rid, s.valuenum AS v, s._e AS e, s._age AS ag, s._prev AS pv
                FROM g_ g ASOF LEFT JOIN s_ s
                  ON g.{idcol} = s.{idcol} AND g._t >= s._e
            """).df()
        else:
            m = pd.merge_asof(
                left.sort_values("_t"),
                sub.rename(columns={"valuenum": "v", "_e": "e", "_age": "ag", "_prev": "pv"}
                           ).sort_values("e"),
                left_on="_t", right_on="e", by=idcol,
                direction="backward")[["_rid", "v", "e", "ag", "pv"]]

        m = m.set_index("_rid").reindex(out["_rid"].to_numpy())
        eligible = out["_t"].to_numpy() - m["e"].to_numpy()   # time since the value became usable
        stale = out["_t"].to_numpy() - m["ag"].to_numpy()      # age of the specimen itself
        ok = np.isfinite(eligible) & (stale <= max_stale)
        out[f"{a}_val_{tag}"]   = np.where(ok, m["v"].to_numpy(), np.nan).astype("float32")
        out[f"{a}_stale_{tag}"] = np.where(ok, stale, np.nan).astype("float32")
        out[f"{a}_delta_{tag}"] = np.where(
            ok, m["v"].to_numpy() - m["pv"].to_numpy(), np.nan).astype("float32")

    return out.drop(columns=["_t", "_rid"])


def build_two_clock(grid, labs, idcol, obs_col, avail_col, name):
    p_obs   = DIR["cache"] / f"feat_obs_{name}.parquet"
    p_avail = DIR["cache"] / f"feat_avail_{name}.parquet"
    if cached(p_obs) and cached(p_avail):
        print(f"[{name}] loaded from cache")
        return pd.read_parquet(p_obs), pd.read_parquet(p_avail)

    lb = labs[labs["analyte"].isin(CFG.lab_panel)].copy()
    lb[obs_col] = lb[obs_col].astype("float64")
    lb[avail_col] = lb[avail_col].astype("float64")
    lb = lb[(lb[obs_col] >= -6) & (lb[obs_col] <= CFG.horizon_hours + 1)]

    with Stage(f"features-obs-{name}"):
        f_obs = asof_features(grid, lb, idcol, obs_col, CFG.lab_panel, "obs",
                              CFG.max_staleness_hours, age_col=obs_col)
    with Stage(f"features-avail-{name}"):
        f_av = asof_features(grid, lb, idcol, avail_col, CFG.lab_panel, "avail",
                             CFG.max_staleness_hours, age_col=obs_col)

    write_parquet(f_obs, f"feat_obs_{name}")
    write_parquet(f_av, f"feat_avail_{name}")
    return f_obs, f_av


# observation clock = hours from ICU admission to draw; availability clock adds the latency
labs_m["obs_h"]   = labs_m["hours_from_icu_admit"].astype("float64")
labs_m["avail_h"] = labs_m["obs_h"] + labs_m["latency_h"]
feat_obs_m, feat_av_m = build_two_clock(grid_m, labs_m, "stay_id", "obs_h", "avail_h", "mimic")

print(f"\nMIMIC-IV feature matrices: {feat_obs_m.shape[0]:,} rows x "
      f"{feat_obs_m.shape[1]-2} observation-clock features")

[mimic] loaded from cache

MIMIC-IV feature matrices: 823,501 rows x 54 observation-clock features


In [15]:
if CFG.include_eicu:
    labs_e["obs_h"]   = labs_e["hours_from_icu_admit"].astype("float64")
    labs_e["avail_h"] = labs_e["obs_h"] + labs_e["latency_h"]
    labs_e = labs_e.rename(columns={"labresult": "valuenum"}) if "valuenum" not in labs_e.columns else labs_e
    feat_obs_e, feat_av_e = build_two_clock(grid_e, labs_e, "patientunitstayid",
                                            "obs_h", "avail_h", "eicu")
    print(f"eICU feature matrices: {feat_obs_e.shape[0]:,} rows")
else:
    feat_obs_e = feat_av_e = pd.DataFrame()

[eicu] loaded from cache
eICU feature matrices: 1,892,310 rows


In [16]:
# ---- how far apart are the two matrices? ----------------------------------
# A direct measure of the deployment gap at the feature level, before any model sees it.
rows = []
for a in CFG.lab_panel:
    co, ca = f"{a}_val_obs", f"{a}_val_avail"
    if co not in feat_obs_m.columns:
        continue
    vo, va = feat_obs_m[co], feat_av_m[ca]
    present_o, present_a = vo.notna(), va.notna()
    both = present_o & present_a
    differs = both & (~np.isclose(vo.fillna(0), va.fillna(0), equal_nan=False))
    lost = present_o & (~present_a)
    so = feat_obs_m[f"{a}_stale_obs"]
    sa = feat_av_m[f"{a}_stale_avail"]
    rows.append({
        "analyte": a,
        "hours with a value, obs clock %": round(100 * present_o.mean(), 2),
        "hours with a value, avail clock %": round(100 * present_a.mean(), 2),
        "value absent under avail clock %": round(100 * lost.mean(), 2),
        "value differs %": round(100 * differs.mean(), 2),
        "median staleness obs (h)": round(float(so.median()), 2),
        "median staleness avail (h)": round(float(sa.median()), 2),
    })

gap = pd.DataFrame(rows).sort_values("value differs %", ascending=False).reset_index(drop=True)
gap["discordant %"] = (gap["value absent under avail clock %"] + gap["value differs %"]).round(2)
display(gap)
write_table(gap, "t25_feature_level_clock_gap")

print(f"\nAcross the panel, a median of {gap['discordant %'].median():.1f}% of scoring hours carry a")
print("different value under the availability clock than under the observation clock.")
print(f"Median staleness rises from {gap['median staleness obs (h)'].median():.2f} h to "
      f"{gap['median staleness avail (h)'].median():.2f} h.")

,analyte,"hours with a value, obs clock %","hours with a value, avail clock %",value absent under avail clock %,value differs %,median staleness obs (h),median staleness avail (h),discordant %
0,glucose,97.190,96.510,0.680,8.250,7.510,8.440,8.930
1,potassium,96.500,95.690,0.820,6.940,7.280,8.240,7.760
2,sodium,96.470,95.650,0.820,6.470,7.270,8.220,7.290
3,chloride,96.470,95.650,0.820,6.350,7.370,8.320,7.170
4,urea_nitrogen,96.900,96.170,0.730,6.190,7.580,8.520,6.920
5,anion_gap,96.320,95.470,0.850,6.160,7.570,8.520,7.010
6,bicarbonate,96.360,95.520,0.840,5.960,7.570,8.500,6.800
7,creatinine,96.900,96.160,0.740,5.040,7.550,8.500,5.780
8,hemoglobin,96.850,96.400,0.450,4.400,8.100,8.650,4.850
9,platelet,96.580,96.080,0.500,4.010,8.170,8.730,4.510



Across the panel, a median of 4.7% of scoring hours carry a
different value under the availability clock than under the observation clock.
Median staleness rises from 8.13 h to 8.69 h.


---
## 7. Vital sign features and bedside comparator scores

Vital features are identical under both clocks by construction. NEWS2 and qSOFA are computed from
them, giving a rule-based arm whose behaviour cannot change between clocks. Any narrowing of the gap
between the learned model and the bedside score under the availability clock is therefore attributable
to the laboratory features alone.

In [17]:
def vitals_features(grid, vitals, idcol, window):
    """Last value plus rolling window extrema, on the hourly grid."""
    wide = vitals.pivot_table(index=[idcol, "hour"], columns="vital",
                              values=["mean_v", "min_v", "max_v"], aggfunc="first")
    wide.columns = [f"{v}_{s}" for s, v in wide.columns]
    wide = wide.reset_index()

    out = (grid[[idcol, "hour"]].merge(wide, on=[idcol, "hour"], how="left")
           .sort_values([idcol, "hour"]).reset_index(drop=True))
    val_cols = [c for c in out.columns if c not in (idcol, "hour")]

    ff = out.groupby(idcol)[val_cols].ffill()
    for c in val_cols:
        out[c] = ff[c].astype("float32")

    # MIMIC-IV charts most temperatures in Fahrenheit. Without this the NEWS2
    # temperature component is missing for the majority of scoring hours.
    for s in ("mean_v", "min_v", "max_v"):
        cf, cc = f"temp_f_{s}", f"temp_c_{s}"
        if cf in out.columns:
            conv = ((out[cf] - 32) * 5.0 / 9.0).astype("float32")
            out[cc] = out[cc].fillna(conv) if cc in out.columns else conv

    g = out.groupby(idcol)   # rebuilt after the forward fill
    # rolling extrema over the recent window, which is what a bedside score reacts to
    for base in ("hr", "sbp", "map", "rr", "spo2", "temp_c"):
        mn, mx = f"{base}_min_v", f"{base}_max_v"
        if mn in out.columns:
            out[f"{base}_min_{window}h"] = (g[mn].rolling(window, min_periods=1).min()
                                            .reset_index(level=0, drop=True).astype("float32"))
        if mx in out.columns:
            out[f"{base}_max_{window}h"] = (g[mx].rolling(window, min_periods=1).max()
                                            .reset_index(level=0, drop=True).astype("float32"))
    return out.reset_index(drop=True)


with Stage("vitals-features"):
    vfeat_m = vitals_features(grid_m, vit_m, "stay_id", CFG.vitals_window_hours)
    if CFG.include_eicu:
        vfeat_e = vitals_features(grid_e, vit_e, "patientunitstayid", CFG.vitals_window_hours)

print(f"MIMIC-IV vital features: {vfeat_m.shape[1]-2} columns")
cov = vfeat_m.drop(columns=["stay_id", "hour"]).notna().mean().sort_values(ascending=False)
display(cov.head(20).rename("coverage").to_frame())

[vitals-features] start
[vitals-features] done in 2.07 min
MIMIC-IV vital features: 48 columns


,coverage
hr_max_v,0.994
hr_min_v,0.994
hr_max_6h,0.994
hr_min_6h,0.994
hr_mean_v,0.994
spo2_max_6h,0.992
spo2_min_6h,0.992
spo2_max_v,0.992
spo2_mean_v,0.992
spo2_min_v,0.992


In [18]:
def news2(df, tempc_col="temp_c_mean_v"):
    """NEWS2 from vital sign components. Consciousness is scored from GCS where present."""
    def band(x, cuts, pts):
        s = pd.Series(np.nan, index=x.index, dtype="float32")
        for (lo, hi), p in zip(cuts, pts):
            s = s.mask(x.between(lo, hi, inclusive="left"), p)
        return s

    rr = df.get("rr_mean_v"); spo2 = df.get("spo2_mean_v")
    sbp = df.get("sbp_mean_v"); hr = df.get("hr_mean_v"); tc = df.get(tempc_col)

    parts = {}
    if rr is not None:
        parts["rr"] = band(rr, [(-np.inf, 9), (9, 12), (12, 21), (21, 25), (25, np.inf)], [3, 1, 0, 2, 3])
    if spo2 is not None:
        parts["spo2"] = band(spo2, [(-np.inf, 92), (92, 94), (94, 96), (96, np.inf)], [3, 2, 1, 0])
    if sbp is not None:
        parts["sbp"] = band(sbp, [(-np.inf, 91), (91, 101), (101, 111), (111, 220), (220, np.inf)],
                            [3, 2, 1, 0, 3])
    if hr is not None:
        parts["hr"] = band(hr, [(-np.inf, 41), (41, 51), (51, 91), (91, 111), (111, 131), (131, np.inf)],
                           [3, 1, 0, 1, 2, 3])
    if tc is not None:
        parts["temp"] = band(tc, [(-np.inf, 35.1), (35.1, 36.1), (36.1, 38.1), (38.1, 39.1), (39.1, np.inf)],
                             [3, 1, 0, 1, 2])
    if "gcs_total" in df.columns:
        parts["cons"] = (df["gcs_total"] < 15).astype("float32") * 3

    P = pd.DataFrame(parts)
    return P.sum(axis=1, min_count=1).astype("float32"), P.notna().sum(axis=1).astype("int8")


def qsofa(df):
    rr = df.get("rr_mean_v"); sbp = df.get("sbp_mean_v")
    s = pd.Series(0.0, index=df.index, dtype="float32")
    n = pd.Series(0, index=df.index, dtype="int8")
    if rr is not None:
        s += (rr >= 22).astype("float32"); n += rr.notna().astype("int8")
    if sbp is not None:
        s += (sbp <= 100).astype("float32"); n += sbp.notna().astype("int8")
    if "gcs_total" in df.columns:
        s += (df["gcs_total"] < 15).astype("float32"); n += df["gcs_total"].notna().astype("int8")
    return s.astype("float32"), n


def add_gcs_total(vfeat, vitals, idcol):
    comps = ["gcs_eye", "gcs_verbal", "gcs_motor"]
    have = [c for c in comps if c in set(vitals["vital"])]
    if len(have) < 3:
        return vfeat
    w = (vitals[vitals["vital"].isin(have)]
         .pivot_table(index=[idcol, "hour"], columns="vital", values="mean_v", aggfunc="first"))
    w["gcs_total"] = w[have].sum(axis=1, min_count=3)
    vfeat = vfeat.merge(w[["gcs_total"]].reset_index(), on=[idcol, "hour"], how="left")
    vfeat["gcs_total"] = (vfeat.sort_values([idcol, "hour"]).groupby(idcol)["gcs_total"]
                          .ffill().reset_index(level=0, drop=True).astype("float32"))
    return vfeat


vfeat_m = add_gcs_total(vfeat_m, vit_m, "stay_id")

# The comparator must be keyed off the frame the scores were computed from.
# vitals_features re-sorts its output, so taking the keys from the grid instead
# would pair every score with the wrong patient-hour.
comp_m = vfeat_m[["stay_id", "hour"]].copy()
comp_m["news2"], comp_m["news2_components"] = news2(vfeat_m)
comp_m["qsofa"], comp_m["qsofa_components"] = qsofa(vfeat_m)

# eICU carries no GCS, so its NEWS2 has no consciousness component. A matched
# variant that omits consciousness in both databases makes the cross-database
# contrast like for like; the full score is retained for the internal analysis.
vm_nc = vfeat_m.drop(columns=["gcs_total"], errors="ignore")
comp_m["news2_nc"], _ = news2(vm_nc)
comp_m["qsofa_nc"], _ = qsofa(vm_nc)

assert len(comp_m) == len(grid_m), "comparator and grid must cover the same scoring hours"
write_parquet(comp_m, "comparator_mimic")

if CFG.include_eicu:
    comp_e = vfeat_e[["patientunitstayid", "hour"]].copy()
    comp_e["news2"], comp_e["news2_components"] = news2(vfeat_e)
    comp_e["qsofa"], comp_e["qsofa_components"] = qsofa(vfeat_e)
    comp_e["news2_nc"] = comp_e["news2"]   # eICU has no consciousness component to remove
    comp_e["qsofa_nc"] = comp_e["qsofa"]
    assert len(comp_e) == len(grid_e)
    write_parquet(comp_e, "comparator_eicu")

print(f"\nMatched comparator (no consciousness component), MIMIC-IV:")
print(f"  full NEWS2 median {comp_m['news2'].median():.0f}, "
      f"matched NEWS2 median {comp_m['news2_nc'].median():.0f}")

print("NEWS2 distribution, MIMIC-IV:")
display(comp_m["news2"].describe(percentiles=[.25, .5, .75, .9, .99]))
print(f"Scoring hours with a computable NEWS2: {100*comp_m['news2'].notna().mean():.1f}%")
print(f"Median components contributing: {comp_m['news2_components'].median():.0f} of 6")
print(f"\nqSOFA >= 2 in {100*(comp_m['qsofa'] >= 2).mean():.2f}% of scoring hours")


Matched comparator (no consciousness component), MIMIC-IV:
  full NEWS2 median 3, matched NEWS2 median 2
NEWS2 distribution, MIMIC-IV:


count   823,501.000
mean          3.173
std           2.442
min           0.000
25%           1.000
50%           3.000
75%           5.000
90%           7.000
99%          10.000
max          18.000
Name: news2, dtype: float64

Scoring hours with a computable NEWS2: 100.0%
Median components contributing: 6 of 6

qSOFA >= 2 in 13.06% of scoring hours


---
## 8. Design matrix assembly

Each clock produces one design matrix. Both share identical rows, identical labels, identical vital
sign features and identical static covariates. They differ only in the laboratory block, which is
what makes the comparison clean: any difference in model behaviour is attributable to the clock and
to nothing else.

In [19]:
STATIC_DESC_M = ["first_careunit", "admission_type", "anchor_year_group"]
STATIC_DESC_E = ["unittype", "region", "numbedscategory"]

def assemble(grid, feats, vfeat, comp, cohort, idcol, desc_cols):
    X = grid[[idcol, "hour", "y", "first_event_h", "hours_to_event"]].copy()
    X = X.merge(feats, on=[idcol, "hour"], how="left")
    X = X.merge(vfeat, on=[idcol, "hour"], how="left")
    X = X.merge(comp[[idcol, "hour", "news2", "qsofa"]], on=[idcol, "hour"], how="left")

    keep = [c for c in (["age_years", "gender"] + desc_cols) if c in cohort.columns]
    X = X.merge(cohort[[idcol] + keep], on=idcol, how="left")

    # Numeric encodings for the model. Descriptive columns are carried for stratification
    # in Part 3 and are deliberately not model inputs, because a unit label or a hospital
    # region does not transport between databases.
    X["age_years"] = pd.to_numeric(X.get("age_years"), errors="coerce").astype("float32")
    gen = X["gender"] if "gender" in X.columns else pd.Series(index=X.index, dtype=object)
    X["is_female"] = gen.astype(str).str.upper().str.startswith("F").astype("float32")
    return X


with Stage("assemble-mimic"):
    Xo_m = assemble(grid_m, feat_obs_m, vfeat_m, comp_m, cohort_m, "stay_id", STATIC_DESC_M)
    Xa_m = assemble(grid_m, feat_av_m,  vfeat_m, comp_m, cohort_m, "stay_id", STATIC_DESC_M)

if CFG.include_eicu and len(grid_e):
    with Stage("assemble-eicu"):
        Xo_e = assemble(grid_e, feat_obs_e, vfeat_e, comp_e, cohort_e,
                        "patientunitstayid", STATIC_DESC_E)
        Xa_e = assemble(grid_e, feat_av_e,  vfeat_e, comp_e, cohort_e,
                        "patientunitstayid", STATIC_DESC_E)
else:
    Xo_e = Xa_e = pd.DataFrame()

del feat_obs_m, feat_av_m
gc.collect()

# ---------------------------------------------------------------------------
# A canonical feature space, identical across both databases and both clocks.
# Any column a database does not supply is introduced as all-missing; the gradient
# boosted trees handle missingness natively, and holding the space fixed is what
# makes the external cohort and the two clocks directly comparable.
# ---------------------------------------------------------------------------
VITAL_BASES = ["hr", "sbp", "dbp", "map", "rr", "spo2", "temp_c"]
VITAL_FEATURES = ([f"{b}_{s}" for b in VITAL_BASES for s in ("mean_v", "min_v", "max_v")] +
                  [f"{b}_{s}_{CFG.vitals_window_hours}h" for b in VITAL_BASES for s in ("min", "max")] +
                  ["gcs_total"])
STATIC_FEATURES = ["age_years", "is_female", "hour"]

def clock_features(tag):
    return [f"{a}_{s}_{tag}" for a in CFG.lab_panel for s in ("val", "stale", "delta")]

LAB_OBS, LAB_AV = clock_features("obs"), clock_features("avail")
FEAT_OBS = LAB_OBS + VITAL_FEATURES + STATIC_FEATURES
FEAT_AV  = LAB_AV  + VITAL_FEATURES + STATIC_FEATURES
assert len(FEAT_OBS) == len(FEAT_AV), "clock matrices must have matching width"

def design(X, feats):
    """Reindex onto the canonical space and coerce to a numeric matrix."""
    D = X.reindex(columns=feats)
    for c in feats:
        if D[c].dtype.name != "float32":
            D[c] = pd.to_numeric(D[c], errors="coerce").astype("float32")
    return D

print(f"Design matrix: {len(Xo_m):,} rows, {len(FEAT_OBS)} features")
print(f"  laboratory : {len(LAB_OBS)}  ({len(CFG.lab_panel)} analytes x value, staleness, delta)")
print(f"  vital sign : {len(VITAL_FEATURES)}")
print(f"  static     : {len(STATIC_FEATURES)}")
print(f"Event rate: {100*Xo_m['y'].mean():.2f}% of scoring hours")

if len(Xo_e):
    miss_e = [c for c in FEAT_OBS if c not in Xo_e.columns]
    print(f"\nColumns absent from eICU and supplied as missing: {len(miss_e)}")
    if miss_e:
        print("  " + ", ".join(miss_e[:12]) + (" ..." if len(miss_e) > 12 else ""))

[assemble-mimic] start
[assemble-mimic] done in 6.0 s
[assemble-eicu] start
[assemble-eicu] done in 9.3 s
Design matrix: 823,501 rows, 93 features
  laboratory : 54  (18 analytes x value, staleness, delta)
  vital sign : 36
  static     : 3
Event rate: 3.00% of scoring hours

Columns absent from eICU and supplied as missing: 3
  dbp_min_6h, dbp_max_6h, gcs_total


---
## 9. Model

A single gradient boosted classifier, fixed hyperparameters, no tuning. The study is not about
squeezing out discrimination and a tuned model would only make the clock comparison harder to read.
The split is temporal within MIMIC-IV, using the de-identified era label rather than the shifted
calendar timestamps, and eICU serves as the external cohort.

Two models are fitted. One sees only observation-clock laboratory features, mirroring how the
literature is built. The other sees only availability-clock features, representing a model developed
with knowledge of the delay. Both are then scored under both clocks.

In [20]:
def era_split(X, cohort, idcol):
    e = cohort[[idcol, "anchor_year_group"]]
    X = X.merge(e, on=idcol, how="left", suffixes=("", "_e"))
    col = "anchor_year_group_e" if "anchor_year_group_e" in X.columns else "anchor_year_group"
    tr = X[col].isin(CFG.train_eras)
    va = X[col].isin(CFG.valid_eras)
    te = X[col].isin(CFG.test_eras)
    return tr.to_numpy(), va.to_numpy(), te.to_numpy()

tr_i, va_i, te_i = era_split(Xo_m, cohort_m, "stay_id")
eras = cohort_m["anchor_year_group"].value_counts()
print("Era distribution in the cohort:")
display(eras.rename("stays").to_frame())

print(f"\ntrain {tr_i.sum():,} hours | valid {va_i.sum():,} | test {te_i.sum():,}")
if tr_i.sum() == 0 or te_i.sum() == 0:
    raise ValueError(
        "Training or test split is empty. The era labels present in this cohort are "
        f"{sorted(eras.index)}; the configured splits are train={CFG.train_eras}, "
        f"valid={CFG.valid_eras}, test={CFG.test_eras}. Adjust the era tuples in Config2.")
print(f"event rate  train {100*Xo_m.loc[tr_i,'y'].mean():.2f}%  "
      f"valid {100*Xo_m.loc[va_i,'y'].mean():.2f}%  test {100*Xo_m.loc[te_i,'y'].mean():.2f}%")

# no patient appears in more than one split because the split is by stay-level era
ids = Xo_m["stay_id"].to_numpy()
assert len(set(ids[tr_i]) & set(ids[te_i])) == 0, "stay leakage between train and test"
print("No stay appears in more than one split.")

Era distribution in the cohort:


,stays
anchor_year_group,
2008 - 2010,15393
2011 - 2013,11732
2014 - 2016,11461
2017 - 2019,10150



train 465,168 hours | valid 178,194 | test 180,139
event rate  train 3.09%  valid 3.11%  test 2.68%
No stay appears in more than one split.


In [21]:
def fit_model(X, feats, tr, va, label):
    """
    Early stopping against the era validation set, which is disjoint at the level of the
    ICU stay. sklearn's internal validation_fraction splits rows at random, which places
    hours from the same stay on both sides of the split; those rows are strongly
    correlated, the internal score keeps improving, and the iteration cap ends up binding
    instead of the validation loss.
    """
    Xtr, ytr = design(X.loc[tr], feats), X.loc[tr, "y"].to_numpy()
    Xva, yva = design(X.loc[va], feats), X.loc[va, "y"].to_numpy()
    scorable = len(Xva) > 0 and len(np.unique(yva)) > 1

    clf = HistGradientBoostingClassifier(
        max_iter=CFG.hgb_max_iter,
        learning_rate=CFG.hgb_learning_rate,
        max_leaf_nodes=CFG.hgb_max_leaf_nodes,
        min_samples_leaf=CFG.hgb_min_samples_leaf,
        l2_regularization=CFG.hgb_l2,
        early_stopping=False, warm_start=True,
        random_state=CFG.seed,
    )

    best_auc, best_iter, stalls, trace = -np.inf, CFG.es_eval_every, 0, []
    with Stage(f"fit-{label}"):
        if scorable:
            for n in range(CFG.es_eval_every, CFG.hgb_max_iter + 1, CFG.es_eval_every):
                clf.set_params(max_iter=n)
                clf.fit(Xtr, ytr)
                auc = roc_auc_score(yva, clf.predict_proba(Xva)[:, 1])
                trace.append({"model": label, "iterations": n, "validation_auroc": round(auc, 5)})
                if auc > best_auc + 1e-5:
                    best_auc, best_iter, stalls = auc, n, 0
                else:
                    stalls += 1
                    if stalls >= CFG.es_patience_evals:
                        break
            # refit cleanly at the selected iteration count
            clf.set_params(max_iter=best_iter, warm_start=False)
            clf.fit(Xtr, ytr)
        else:
            clf.set_params(max_iter=CFG.hgb_max_iter, warm_start=False)
            clf.fit(Xtr, ytr)
            best_iter = clf.n_iter_

    if scorable:
        print(f"  {label}: stopped at {best_iter} iterations "
              f"(searched to {trace[-1]['iterations']}) | validation AUROC {best_auc:.4f}")
        if best_iter >= CFG.hgb_max_iter:
            print("    WARNING: the cap bound rather than the validation curve. Raise hgb_max_iter.")
    else:
        print(f"  {label}: validation era empty or single-class; fitted at the cap")

    return clf, pd.DataFrame(trace)

MODEL_OBS, TRACE_OBS = fit_model(Xo_m, FEAT_OBS, tr_i, va_i, "observation-clock")
MODEL_AV,  TRACE_AV  = fit_model(Xa_m, FEAT_AV,  tr_i, va_i, "availability-clock")

trace = pd.concat([TRACE_OBS, TRACE_AV], ignore_index=True)
if len(trace):
    write_table(trace, "t29_early_stopping_trace")
    fig, ax = plt.subplots(figsize=(7.0, 3.8))
    for m, d in trace.groupby("model"):
        ax.plot(d["iterations"], d["validation_auroc"], lw=1.8, label=m)
    ax.set_xlabel("Boosting iterations")
    ax.set_ylabel("AUROC on the stay-disjoint validation era")
    ax.set_title("Validation curve and selected stopping point")
    ax.legend(frameon=False, fontsize=9)
    save_fig(fig, "f10_validation_curve")

import pickle
for nm, mdl in [("model_obs", MODEL_OBS), ("model_avail", MODEL_AV)]:
    p = DIR["models"] / f"{nm}.pkl"
    with open(p, "wb") as f:
        pickle.dump({"model": mdl, "features": FEAT_OBS if nm == "model_obs" else FEAT_AV,
                     "config_sha256": CFG_HASH, "sklearn": sklearn.__version__}, f)
    PROV.add("model", nm, p)
print("\nModels persisted.")

[fit-observation-clock] start


  File "C:\Users\kruta\anaconda3-v2\Lib\site-packages\joblib\externals\loky\backend\context.py", line 199, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\kruta\anaconda3-v2\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\kruta\anaconda3-v2\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\kruta\anaconda3-v2\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


[fit-observation-clock] done in 77.7 s
  observation-clock: stopped at 150 iterations (searched to 275) | validation AUROC 0.7937
[fit-availability-clock] start
[fit-availability-clock] done in 69.9 s
  availability-clock: stopped at 125 iterations (searched to 250) | validation AUROC 0.7917

Models persisted.


---
## 10. Scoring under every clock combination

The four cells of the design. The comparison that matters is the top row: a model built on
observation-clock features, scored the way the literature scores it, and then scored the way it would
actually run.

In [22]:
def score_all(Xo, Xa, idcol, mask, cohort_tag):
    rows = []
    for mname, mdl, mfeat in [("trained_obs", MODEL_OBS, FEAT_OBS),
                              ("trained_avail", MODEL_AV, FEAT_AV)]:
        for cname, Xc in [("scored_obs", Xo), ("scored_avail", Xa)]:
            # rename the clock suffix so the fitted feature names line up
            src_tag = "obs" if cname == "scored_obs" else "avail"
            dst_tag = "obs" if mname == "trained_obs" else "avail"
            cols = {}
            for f in mfeat:
                if f.endswith(f"_{dst_tag}"):
                    cols[f] = f[: -len(dst_tag)] + src_tag
                else:
                    cols[f] = f
            Xin = Xc.loc[mask].reindex(columns=[cols[f] for f in mfeat])
            Xin.columns = mfeat
            Xin = design(Xin, mfeat)
            p = mdl.predict_proba(Xin)[:, 1]
            rows.append({"cohort": cohort_tag, "model": mname, "clock": cname,
                         "p": p, "index": Xc.loc[mask].index.to_numpy()})
    return rows


def preds_frame(Xo, Xa, idcol, mask, cohort_tag, comp):
    base = Xo.loc[mask, [idcol, "hour", "y", "first_event_h", "hours_to_event"]].copy()
    base = base.reset_index(drop=True)
    for r in score_all(Xo, Xa, idcol, mask, cohort_tag):
        base[f"p_{r['model']}__{r['clock']}"] = r["p"].astype("float32")
    comp_cols = [c for c in ["news2", "qsofa", "news2_nc", "qsofa_nc"] if c in comp.columns]
    base = base.merge(comp[[idcol, "hour"] + comp_cols], on=[idcol, "hour"], how="left")    
    base["cohort"] = cohort_tag
    return base


with Stage("score-internal-test"):
    preds_m = preds_frame(Xo_m, Xa_m, "stay_id", te_i, "MIMIC-IV test", comp_m)
    write_parquet(preds_m, "preds_mimic_test")

if CFG.include_eicu:
    with Stage("score-external"):
        mask_e = np.ones(len(Xo_e), dtype=bool)
        preds_e = preds_frame(Xo_e, Xa_e, "patientunitstayid", mask_e, "eICU-CRD external", comp_e)
        write_parquet(preds_e, "preds_eicu_external")

print(f"MIMIC-IV test predictions: {len(preds_m):,} scoring hours")
if CFG.include_eicu:
    print(f"eICU external predictions: {len(preds_e):,} scoring hours")

[score-internal-test] start
[score-internal-test] done in 4.1 s
[score-external] start
[score-external] done in 32.4 s
MIMIC-IV test predictions: 180,139 scoring hours
eICU external predictions: 1,892,310 scoring hours


In [23]:
# ---- discrimination under each clock combination --------------------------
def metrics_block(df, tag):
    out = []
    y = df["y"].to_numpy()
    for col in [c for c in df.columns if c.startswith("p_")]:
        p = df[col].to_numpy()
        m, c = col[2:].split("__")
        out.append({"cohort": tag, "model": m, "clock": c,
                    "AUROC": roc_auc_score(y, p),
                    "AUPRC": average_precision_score(y, p),
                    "Brier": brier_score_loss(y, p)})
    for col, nm in [("news2", "NEWS2"), ("qsofa", "qSOFA"),
                    ("news2_nc", "NEWS2 (matched, no consciousness)"),
                    ("qsofa_nc", "qSOFA (matched, no consciousness)")]:
        if col not in df.columns:
            continue
        s = df[col]
        ok = s.notna().to_numpy()
        if ok.sum() > 1000 and len(np.unique(y[ok])) > 1:
            out.append({"cohort": tag, "model": nm, "clock": "latency-insensitive",
                        "AUROC": roc_auc_score(y[ok], s[ok].to_numpy()),
                        "AUPRC": average_precision_score(y[ok], s[ok].to_numpy()),
                        "Brier": np.nan})
    return pd.DataFrame(out)

perf = metrics_block(preds_m, "MIMIC-IV test")
if CFG.include_eicu:
    perf = pd.concat([perf, metrics_block(preds_e, "eICU-CRD external")], ignore_index=True)

perf[["AUROC", "AUPRC", "Brier"]] = perf[["AUROC", "AUPRC", "Brier"]].round(4)
display(perf)
write_table(perf, "t26_discrimination_by_clock")

# the headline gap
def gap_for(tag):
    d = perf[(perf["cohort"] == tag) & (perf["model"] == "trained_obs")]
    a = d.loc[d["clock"] == "scored_obs", "AUROC"]
    b = d.loc[d["clock"] == "scored_avail", "AUROC"]
    if len(a) and len(b):
        return float(a.iloc[0]), float(b.iloc[0])
    return np.nan, np.nan

for tag in perf["cohort"].unique():
    a, b = gap_for(tag)
    news = perf[(perf["cohort"] == tag) & (perf["model"] == "NEWS2")]["AUROC"]
    print(f"\n{tag}")
    print(f"  Model trained on the observation clock, scored on the observation clock: AUROC {a:.4f}")
    print(f"  The same model scored on the availability clock:                         AUROC {b:.4f}")
    print(f"  Discrimination attributable to results not yet visible:                  {a-b:+.4f}")
    if len(news):
        n = float(news.iloc[0])
        print(f"  NEWS2 (latency-insensitive):                                            AUROC {n:.4f}")
        if np.isfinite(a) and np.isfinite(b):
            m_obs, m_av = a - n, b - n
            msg = f"  Margin over NEWS2 falls from {m_obs:+.4f} to {m_av:+.4f}"
            if m_obs > 0.005:
                msg += f", a {100*(1 - m_av/m_obs):.1f}% reduction in the apparent advantage."
            else:
                msg += "; the margin is too small for a proportional reduction to be meaningful."
            print(msg)

,cohort,model,clock,AUROC,AUPRC,Brier
0,MIMIC-IV test,trained_obs,scored_obs,0.824,0.305,0.022
1,MIMIC-IV test,trained_obs,scored_avail,0.822,0.299,0.022
2,MIMIC-IV test,trained_avail,scored_obs,0.824,0.293,0.022
3,MIMIC-IV test,trained_avail,scored_avail,0.823,0.298,0.022
4,MIMIC-IV test,NEWS2,latency-insensitive,0.612,0.045,NaN
5,MIMIC-IV test,qSOFA,latency-insensitive,0.591,0.036,NaN
6,MIMIC-IV test,"NEWS2 (matched, no consciousness)",latency-insensitive,0.617,0.044,NaN
7,MIMIC-IV test,"qSOFA (matched, no consciousness)",latency-insensitive,0.576,0.034,NaN
8,eICU-CRD external,trained_obs,scored_obs,0.717,0.068,0.022
9,eICU-CRD external,trained_obs,scored_avail,0.711,0.065,0.022



MIMIC-IV test
  Model trained on the observation clock, scored on the observation clock: AUROC 0.8241
  The same model scored on the availability clock:                         AUROC 0.8222
  Discrimination attributable to results not yet visible:                  +0.0019
  NEWS2 (latency-insensitive):                                            AUROC 0.6124
  Margin over NEWS2 falls from +0.2117 to +0.2098, a 0.9% reduction in the apparent advantage.

eICU-CRD external
  Model trained on the observation clock, scored on the observation clock: AUROC 0.7171
  The same model scored on the availability clock:                         AUROC 0.7107
  Discrimination attributable to results not yet visible:                  +0.0064
  NEWS2 (latency-insensitive):                                            AUROC 0.5794
  Margin over NEWS2 falls from +0.1377 to +0.1313, a 4.6% reduction in the apparent advantage.


In [24]:
# ---- where the gap sits: by scoring hour ----------------------------------
# Part 1 showed the record is most incomplete early. If the mechanism is what we claim,
# the discrimination gap should follow the same shape.
def gap_by_hour(df, tag, bins=((1, 6), (7, 12), (13, 24), (25, 36), (37, 48))):
    out = []
    for lo, hi in bins:
        s = df[(df["hour"] >= lo) & (df["hour"] <= hi)]
        if len(s) < 2000 or s["y"].nunique() < 2:
            continue
        a = roc_auc_score(s["y"], s["p_trained_obs__scored_obs"])
        b = roc_auc_score(s["y"], s["p_trained_obs__scored_avail"])
        out.append({"cohort": tag, "hours": f"{lo}-{hi}", "n": len(s),
                    "events": int(s["y"].sum()),
                    "AUROC obs clock": round(a, 4), "AUROC avail clock": round(b, 4),
                    "gap": round(a - b, 4)})
    return pd.DataFrame(out)

gh = gap_by_hour(preds_m, "MIMIC-IV test")
if CFG.include_eicu:
    gh = pd.concat([gh, gap_by_hour(preds_e, "eICU-CRD external")], ignore_index=True)
display(gh)
write_table(gh, "t27_clock_gap_by_hour")

fig, ax = plt.subplots(figsize=(7.0, 4.0))
for tag, mk in [("MIMIC-IV test", "o"), ("eICU-CRD external", "s")]:
    s = gh[gh["cohort"] == tag]
    if len(s):
        ax.plot(s["hours"], s["gap"], marker=mk, lw=2, label=tag)
ax.axhline(0, color="grey", lw=0.9, ls=":")
ax.set_xlabel("Scoring hour after ICU admission")
ax.set_ylabel("AUROC, observation clock minus availability clock")
ax.set_title("Discrimination attributable to results not yet visible")
ax.legend(frameon=False, fontsize=9)
save_fig(fig, "f9_clock_gap_by_hour")

,cohort,hours,n,events,AUROC obs clock,AUROC avail clock,gap
0,MIMIC-IV test,1-6,6098,470,0.848,0.848,-0.001
1,MIMIC-IV test,7-12,34941,1440,0.797,0.795,0.002
2,MIMIC-IV test,13-24,60076,1527,0.811,0.809,0.002
3,MIMIC-IV test,25-36,44700,801,0.809,0.809,0.001
4,MIMIC-IV test,37-48,34324,582,0.809,0.803,0.006
5,eICU-CRD external,1-6,67167,2913,0.703,0.699,0.003
6,eICU-CRD external,7-12,395212,12175,0.707,0.702,0.004
7,eICU-CRD external,13-24,665357,13506,0.699,0.690,0.009
8,eICU-CRD external,25-36,445636,6333,0.704,0.696,0.008
9,eICU-CRD external,37-48,318938,4207,0.693,0.687,0.006


WindowsPath('C:/Research_Paper_2/result_availability_audit/figures/f9_clock_gap_by_hour.png')

---
## 11. Checks before Part 3

In [25]:
checks = []
a_int, b_int = gap_for("MIMIC-IV test")

checks.append(("Escalation events resolved in both databases",
               bool(events_m["first_event_h"].notna().sum() > 1000 and
                    (not CFG.include_eicu or events_e["first_event_h"].notna().sum() > 1000)),
               f"MIMIC {int(events_m['first_event_h'].notna().sum()):,}" +
               (f", eICU {int(events_e['first_event_h'].notna().sum()):,}" if CFG.include_eicu else "")))

checks.append(("Event rate is in a workable range (0.2-10% of hours)",
               bool(0.002 < Xo_m["y"].mean() < 0.10),
               f"{100*Xo_m['y'].mean():.2f}% of scoring hours"))

checks.append(("The two clock matrices genuinely differ",
               bool(gap["discordant %"].median() > 1.0),
               f"median {gap['discordant %'].median():.1f}% of hours discordant"))

checks.append(("Staleness increases under the availability clock",
               bool(gap["median staleness avail (h)"].median() >
                    gap["median staleness obs (h)"].median()),
               f"{gap['median staleness obs (h)'].median():.2f} h -> "
               f"{gap['median staleness avail (h)'].median():.2f} h"))

checks.append(("Model discriminates above chance",
               bool(np.isfinite(a_int) and a_int > 0.70),
               f"AUROC {a_int:.4f} on the observation clock"))

checks.append(("A clock gap is present and in the expected direction",
               bool(np.isfinite(a_int) and np.isfinite(b_int) and a_int > b_int),
               f"{a_int-b_int:+.4f} AUROC"))

checks.append(("Comparator is computable on most scoring hours",
               bool(comp_m["news2"].notna().mean() > 0.80),
               f"NEWS2 available for {100*comp_m['news2'].notna().mean():.1f}% of hours"))

res = pd.DataFrame(checks, columns=["check", "passed", "observed"])
res["status"] = np.where(res["passed"], "PASS", "REVIEW")
display(res[["check", "status", "observed"]])
write_table(res, "t28_part2_gate_checks")

if res["passed"].all():
    print("\nAll checks passed. Part 3 can compute lead time, delayed and missed detections,")
    print("and the alert burden comparison from the cached prediction files.")
else:
    print("\nReview the following before Part 3:")
    for _, r in res[~res["passed"]].iterrows():
        print(f"  - {r['check']}: {r['observed']}")

,check,status,observed
0,Escalation events resolved in both databases,PASS,"MIMIC 23,161, eICU 31,021"
1,Event rate is in a workable range (0.2-10% of ...,PASS,3.00% of scoring hours
2,The two clock matrices genuinely differ,PASS,median 4.7% of hours discordant
3,Staleness increases under the availability clock,PASS,8.13 h -> 8.69 h
4,Model discriminates above chance,PASS,AUROC 0.8241 on the observation clock
5,A clock gap is present and in the expected dir...,PASS,+0.0019 AUROC
6,Comparator is computable on most scoring hours,PASS,NEWS2 available for 100.0% of hours



All checks passed. Part 3 can compute lead time, delayed and missed detections,
and the alert burden comparison from the cached prediction files.


In [26]:
manifest = PROV.manifest()
manifest["part1_terminal_chain"] = P1_CHAIN
manifest["part2_summary"] = {
    "mimic_scoring_hours": int(len(Xo_m)),
    "mimic_event_rate": float(Xo_m["y"].mean()),
    "mimic_stays_with_escalation": int(events_m["first_event_h"].notna().sum()),
    "eicu_scoring_hours": int(len(Xo_e)) if CFG.include_eicu else None,
    "eicu_hospitals_retained": int(cohort_e["hospitalid"].nunique()) if CFG.include_eicu else None,
    "n_features": len(FEAT_OBS),
    "lab_panel": list(CFG.lab_panel),
    "median_discordant_pct": float(gap["discordant %"].median()),
    "median_staleness_obs_h": float(gap["median staleness obs (h)"].median()),
    "median_staleness_avail_h": float(gap["median staleness avail (h)"].median()),
    "auroc": perf.set_index(["cohort", "model", "clock"])["AUROC"].to_dict().__class__(
        {f"{c}|{m}|{k}": float(v) for (c, m, k), v in
         perf.set_index(["cohort", "model", "clock"])["AUROC"].items()}),
}

MANIFEST_P = OUT / "part2_manifest.json"
with open(MANIFEST_P, "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(f"Manifest written: {MANIFEST_P}")
print(f"Terminal chain digest: {PROV.chain}")

t = pd.DataFrame([{"stage": k, "seconds": float(v["seconds"])} for k, v in PROV.timings.items()])
if len(t):
    t["minutes"] = (t["seconds"] / 60).round(2)
    display(t.sort_values("seconds", ascending=False).reset_index(drop=True))

if CON is not None:
    CON.close()
print("\nPart 2 complete.")
print("Cached for Part 3: preds_mimic_test, preds_eicu_external, events_*, comparator_*, grid_*")

Manifest written: C:\Research_Paper_2\result_availability_audit\part2_manifest.json
Terminal chain digest: 47ff1b281a4b0ea4e60da8adcf9600a7f763e66be6c0341ed482972e134f8094


,stage,seconds,minutes
0,vitals-features,123.990,2.070
1,fit-observation-clock,77.740,1.300
2,fit-availability-clock,69.890,1.160
3,score-external,32.380,0.540
4,assemble-eicu,9.300,0.160
5,load-part1,6.200,0.100
6,assemble-mimic,5.960,0.100
7,score-internal-test,4.110,0.070
8,vitals-mimic,1.480,0.020
9,grid-build,1.290,0.020



Part 2 complete.
Cached for Part 3: preds_mimic_test, preds_eicu_external, events_*, comparator_*, grid_*
